In [1]:
import torch
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq, 
    TrainerCallback,
    EarlyStoppingCallback
)
from datasets import load_dataset, concatenate_datasets

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model = model.to("cuda")
print("✅ Modelo listo")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Modelo listo


In [3]:
dataset = load_dataset("stanfordnlp/imdb")

train_pos = dataset["train"].filter(lambda x: x["label"] == 1).select(range(1000))
train_neg = dataset["train"].filter(lambda x: x["label"] == 0).select(range(1000))
train_data = concatenate_datasets([train_pos, train_neg]).shuffle(seed=42)

val_pos = dataset["test"].filter(lambda x: x["label"] == 1).select(range(200))
val_neg = dataset["test"].filter(lambda x: x["label"] == 0).select(range(200))
val_data = concatenate_datasets([val_pos, val_neg]).shuffle(seed=42)

print(f"✅ Train: {len(train_data)} | Val: {len(val_data)}")

✅ Train: 2000 | Val: 400


In [4]:
def preprocess(examples):
    inputs = [
        f"Classify the sentiment of this movie review as 'positive' or 'negative':\n\n{text}"
        for text in examples["text"]
    ]
    targets = [
        "positive" if label == 1 else "negative"
        for label in examples["label"]
    ]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        targets,
        max_length=8,
        truncation=True,
        padding="max_length"
    )
    
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = label_ids
    return model_inputs

print("⏳ Preprocesando...")
train_tokenized = train_data.map(preprocess, batched=True, remove_columns=["text", "label"])
val_tokenized = val_data.map(preprocess, batched=True, remove_columns=["text", "label"])
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")
print("✅ Listo")

⏳ Preprocesando...
✅ Listo


In [5]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-sentiment",
    num_train_epochs=10,              # Ponemos más epochs, early stopping decide cuándo parar
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Early stopping monitorea eval_loss
    greater_is_better=False,            # Queremos que el loss BAJE
    predict_with_generate=False,
    fp16=False,
    bf16=True,
    report_to="none",
    learning_rate=3e-4,
    lr_scheduler_type="cosine",         # LR baja gradualmente siguiendo curva coseno
    max_grad_norm=1.0                   # Gradient clipping
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
print("✅ Config lista")
print("   Early Stopping: activado (patience=2)")
print("   LR Scheduler: cosine")
print("   Gradient Clipping: max_norm=1.0")

✅ Config lista
   Early Stopping: activado (patience=2)
   LR Scheduler: cosine
   Gradient Clipping: max_norm=1.0


In [6]:
class LossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            loss = logs.get('loss', 'N/A')
            eval_loss = logs.get('eval_loss', 'N/A')
            epoch = state.epoch
            print(f"Epoch {epoch:.1f} | Step {state.global_step} | Loss: {loss} | Eval Loss: {eval_loss}")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    callbacks=[
        LossCallback(),
        EarlyStoppingCallback(early_stopping_patience=2)  # Para si no mejora en 2 epochs seguidas
    ]
)

print("🚀 Iniciando fine-tuning mejorado...")
print("   Early stopping con patience=2")
print("   Máximo 10 epochs pero parará antes si no mejora\n")
trainer.train()
print("\n✅ Fine-tuning completado!")
print(f"   Mejor modelo guardado automáticamente")

🚀 Iniciando fine-tuning mejorado...
   Early stopping con patience=2
   Máximo 10 epochs pero parará antes si no mejora



c:\Users\pamge\OneDrive\Desktop\clasificador-reseñas-ia\sentiment_env\Lib\site-packages\transformers\data\data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,0.169433,0.247474
2,0.063276,0.174336
3,0.048054,0.145085
4,0.025668,0.163958
5,0.026169,0.192459


Epoch 0.2 | Step 50 | Loss: 0.13733335494995116 | Eval Loss: N/A
Epoch 0.4 | Step 100 | Loss: 0.1719021987915039 | Eval Loss: N/A
Epoch 0.6 | Step 150 | Loss: 0.12647321701049805 | Eval Loss: N/A
Epoch 0.8 | Step 200 | Loss: 0.1828368377685547 | Eval Loss: N/A
Epoch 1.0 | Step 250 | Loss: 0.1694334602355957 | Eval Loss: N/A
Epoch 1.0 | Step 250 | Loss: N/A | Eval Loss: 0.2474742829799652


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 4.2 | Step 1050 | Loss: 0.0051834237575531005 | Eval Loss: N/A
Epoch 4.4 | Step 1100 | Loss: 0.006976359486579895 | Eval Loss: N/A
Epoch 4.6 | Step 1150 | Loss: 0.014341094493865968 | Eval Loss: N/A
Epoch 4.8 | Step 1200 | Loss: 0.0027290281653404237 | Eval Loss: N/A
Epoch 5.0 | Step 1250 | Loss: 0.026169204711914064 | Eval Loss: N/A
Epoch 5.0 | Step 1250 | Loss: N/A | Eval Loss: 0.1924586147069931


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5.0 | Step 1250 | Loss: N/A | Eval Loss: N/A


[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



✅ Fine-tuning completado!
   Mejor modelo guardado automáticamente


In [7]:
model.save_pretrained("./flan-t5-sentiment-model")
tokenizer.save_pretrained("./flan-t5-sentiment-model")
print("✅ Modelo guardado en ./flan-t5-sentiment-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo guardado en ./flan-t5-sentiment-model
